# Financial Inclusion Indicators — FinScope 2024 Rwanda

## Objective
Calculate financial-inclusion indicators using the appropriate survey weights, denominators, and response definitions.

## Analysis plan
1. Load the FinScope 2024 Rwanda dataset.
2. Verify survey weights and identify sampling-design variables.
3. Define financial-inclusion indicators.
4. Calculate weighted national estimates.
5. Calculate weighted district estimates.
6. Assess uncertainty and confidence intervals where the survey-design information permits.
7. Document assumptions and limitations.

In [1]:
import pandas as pd
from pathlib import Path

project_root = Path.cwd().parent

data_path = project_root / "data" / "datasets" / "FinScope2024Rwanda.dta"

df = pd.read_stata(data_path)

print("Dataset shape:", df.shape)
print("Dataset loaded successfully.")

Dataset shape: (13994, 2442)
Dataset loaded successfully.


In [2]:
# Step 4: Inspect survey-related variable labels

reader = pd.io.stata.StataReader(data_path)
variable_labels = reader.variable_labels()

search_terms = [
    "weight",
    "stratum",
    "cluster",
    "sampling",
    "psu",
    "enumeration area",
    "replicate"
]

for term in search_terms:
    matches = [
        (name, label)
        for name, label in variable_labels.items()
        if term in str(name).lower()
        or term in str(label).lower()
    ]

    print(f"\n--- {term.upper()} ---")

    if matches:
        for name, label in matches:
            print(f"{name}: {label}")
    else:
        print("No matching variables found.")


--- WEIGHT ---
hh_wt: Household weight
pop_wt: Individual weight_16+ years old

--- STRATUM ---
No matching variables found.

--- CLUSTER ---
No matching variables found.

--- SAMPLING ---
No matching variables found.

--- PSU ---
No matching variables found.

--- ENUMERATION AREA ---
No matching variables found.

--- REPLICATE ---
No matching variables found.


In [3]:
# Step 5: Validate survey weights

for col in ["pop_wt", "hh_wt"]:
    print(f"\n--- {col} ---")
    print("Data type:", df[col].dtype)
    print("Missing values:", df[col].isna().sum())
    print("Zero or negative weights:", (df[col] <= 0).sum())
    print(df[col].describe())


--- pop_wt ---
Data type: float64
Missing values: 0
Zero or negative weights: 0
count    13994.000000
mean       582.795269
std        526.620197
min         21.902144
25%        236.557792
50%        436.659803
75%        735.574462
max       5526.044689
Name: pop_wt, dtype: float64

--- hh_wt ---
Data type: float64
Missing values: 0
Zero or negative weights: 0
count    13994.000000
mean       255.501692
std         92.118888
min         53.927869
25%        188.021422
50%        257.652974
75%        314.169213
max        711.812239
Name: hh_wt, dtype: float64


In [4]:
# Step 6: Weighted national bank-account ownership estimate

# Convert responses into numeric values: Yes = 1, No = 0
indicator = (
    df["k1"]
    .astype("string")
    .map({"Yes": 1, "No": 0})
    .astype("float64")
)

# Keep respondents with a valid answer and valid weight
valid = indicator.notna() & df["pop_wt"].notna()

# Calculate weighted and unweighted estimates
weighted_rate = (
    (indicator[valid] * df.loc[valid, "pop_wt"]).sum()
    / df.loc[valid, "pop_wt"].sum()
)

unweighted_rate = indicator[valid].mean()

print("Indicator: Own bank account")
print("Valid respondents:", int(valid.sum()))
print("Unweighted sample percentage:", round(unweighted_rate * 100, 2), "%")
print("Weighted population estimate:", round(weighted_rate * 100, 2), "%")

Indicator: Own bank account
Valid respondents: 13994
Unweighted sample percentage: 20.82 %
Weighted population estimate: 20.2 %


In [5]:
# Step 8: Weighted bank-account ownership by district

district_data = df[["a2", "k1", "pop_wt"]].copy()

district_data["owns_bank_account"] = (
    district_data["k1"]
    .astype("string")
    .map({"Yes": 1, "No": 0})
)

district_data = district_data.dropna(
    subset=["a2", "owns_bank_account", "pop_wt"]
)

district_results = (
    district_data
    .groupby("a2", observed=True)
    .apply(
        lambda group: pd.Series({
            "Valid respondents": len(group),
            "Unweighted rate (%)": group["owns_bank_account"].mean() * 100,
            "Weighted rate (%)": (
                (group["owns_bank_account"] * group["pop_wt"]).sum()
                / group["pop_wt"].sum()
            ) * 100
        }),
        include_groups=False
    )
    .sort_values("Weighted rate (%)")
)

district_results = district_results.round(2)

display(district_results)

,Valid respondents,Unweighted rate (%),Weighted rate (%)
a2,,,
Kirehe,464.0,7.76,5.41
Nyaruguru,400.0,4.25,6.15
Gicumbi,512.0,9.38,6.51
Karongi,399.0,10.78,7.19
Ngoma,400.0,7.50,8.39
Ngororero,400.0,11.25,8.72
Rutsiro,400.0,9.50,9.27
Nyamagabe,400.0,10.25,9.34
Gisagara,400.0,10.50,9.53


In [6]:
print("Columns:", district_results.columns.tolist())
print("\nFirst five rows:")
display(district_results.head())

print("\nLowest weighted rates:")
display(district_results.nsmallest(5, "Weighted rate (%)"))

print("\nHighest weighted rates:")
display(district_results.nlargest(5, "Weighted rate (%)"))

Columns: ['Valid respondents', 'Unweighted rate (%)', 'Weighted rate (%)']

First five rows:


,Valid respondents,Unweighted rate (%),Weighted rate (%)
a2,,,
Kirehe,464.0,7.76,5.41
Nyaruguru,400.0,4.25,6.15
Gicumbi,512.0,9.38,6.51
Karongi,399.0,10.78,7.19
Ngoma,400.0,7.50,8.39



Lowest weighted rates:


,Valid respondents,Unweighted rate (%),Weighted rate (%)
a2,,,
Kirehe,464.0,7.76,5.41
Nyaruguru,400.0,4.25,6.15
Gicumbi,512.0,9.38,6.51
Karongi,399.0,10.78,7.19
Ngoma,400.0,7.50,8.39



Highest weighted rates:


,Valid respondents,Unweighted rate (%),Weighted rate (%)
a2,,,
Kicukiro,607.0,62.44,59.51
Nyarugenge,448.0,52.46,54.26
Gasabo,848.0,48.82,47.17
Rwamagana,464.0,26.72,26.25
Musanze,399.0,25.06,23.33


In [7]:
# Step 10: Search project files for survey documentation

from pathlib import Path

for folder in [project_root, project_root / "data"]:
    print(f"\n--- Files under {folder} ---")

    for path in folder.rglob("*"):
        if path.is_file() and any(
            term in path.name.lower()
            for term in [
                "finscope", "methodology", "sampling",
                "questionnaire", "codebook", "technical",
                "weight", "documentation"
            ]
        ):
            print(path.relative_to(project_root))


--- Files under /home/ibrahim/Desktop/Backup/PPioneers/nisr-big-data-hackathon-project ---
data/datasets/FinScope2024Rwanda.dta
data/reports/FinScope-2024-Presantation.pdf
data/reports/Finscope-Final-Report__10_07_2024-1.pdf
data/reports/ddi-documentation-english-119.pdf
data/reports/Finscope consumer 2024 Final QNR 29 June 2025_Shared_NISR.pdf
data/reports/ddi-documentation-english-120.pdf

--- Files under /home/ibrahim/Desktop/Backup/PPioneers/nisr-big-data-hackathon-project/data ---
data/datasets/FinScope2024Rwanda.dta
data/reports/FinScope-2024-Presantation.pdf
data/reports/Finscope-Final-Report__10_07_2024-1.pdf
data/reports/ddi-documentation-english-119.pdf
data/reports/Finscope consumer 2024 Final QNR 29 June 2025_Shared_NISR.pdf
data/reports/ddi-documentation-english-120.pdf


In [8]:
# Step 11: Search survey documentation for sampling-design details

from pathlib import Path

reports_dir = project_root / "data" / "reports"

pdf_files = [
    reports_dir / "ddi-documentation-english-119.pdf",
    reports_dir / "ddi-documentation-english-120.pdf",
    reports_dir / "Finscope-Final-Report__10_07_2024-1.pdf",
]

search_terms = [
    "sampling design",
    "stratified",
    "primary sampling unit",
    "cluster",
    "stratum",
    "confidence interval",
    "individual weight",
    "pop_wt",
]

try:
    import pymupdf
except ImportError:
    pymupdf = None

if pymupdf is None:
    try:
        import fitz
        pymupdf = fitz
    except ImportError:
        print("PDF reader not installed. Run: %pip install pymupdf")

if pymupdf is not None:
    for pdf_path in pdf_files:
        print(f"\n{'=' * 70}")
        print("DOCUMENT:", pdf_path.name)

        if not pdf_path.exists():
            print("File not found.")
            continue

        doc = pymupdf.open(pdf_path)
        matches_found = 0

        for page_number, page in enumerate(doc, start=1):
            page_text = page.get_text()
            text_lower = page_text.lower()

            for term in search_terms:
                position = text_lower.find(term)

                if position != -1:
                    start = max(0, position - 250)
                    end = min(len(page_text), position + 600)

                    print(f"\nPage {page_number} | Search term: {term}")
                    print(page_text[start:end].replace("\n", " "))
                    matches_found += 1

        if matches_found == 0:
            print("No matching terms found.")

        doc.close()


DOCUMENT: ddi-documentation-english-119.pdf

Page 4 | Search term: primary sampling unit
as covered over a 12 months period (October 2023 to October 2024). In order to represent the seasonality in the income and consumption data, the fieldwork was divided into nine nationally representative cycles.  The NISR developed a Master Sample of primary sampling units (PSUs) based on data from the 2022 Rwanda Census of Population and Housing. This master sample was designed to support various national household surveys during the intercensal period, including the EICV. Samples were drawn from each of the country's 30 districts (strata). Within each stratum, the Master Sample PSUs were selected using probability proportional to size (PPS), with the number of households enumerated in the Census serving as the measure of size for each Enumeration Area (EA).  In order to determine the sample size for the EICV7, NISR examined EICV5 data to

Page 4 | Search term: cluster
tal of 1,674 EAs were distri

In [9]:
# Step 12: Search for possible sampling-design identifiers

terms = [
    "psu", "primary sampling", "enumeration area",
    "cluster", "strat", "sample",
    "district", "urban", "sector", "cell",
    "village", "cycle"
]

for term in terms:
    matches = [
        (name, label)
        for name, label in variable_labels.items()
        if term in str(name).lower()
        or term in str(label).lower()
    ]

    print(f"\n--- {term.upper()} ---")
    if matches:
        for name, label in matches:
            print(f"{name}: {label}")
    else:
        print("No matches.")


--- PSU ---
No matches.

--- PRIMARY SAMPLING ---
No matches.

--- ENUMERATION AREA ---
No matches.

--- CLUSTER ---
No matches.

--- STRAT ---
qf1_08: qf1_08. Ever used - Pension fund administrator
qf3_08: qf3_08. Account registered in own name - Pension fund administrator
qf4_08: qf4_08. Do you currently use - Pension fund administrator
qf5_08_1: qf5_08_1. Pension fund administrator-Deposited or withdrew money
qf5_08_2: qf5_08_2. Pension fund administrator-Bought airtime top-ups or paid a mobile pho
qf5_08_3: qf5_08_3. Pension fund administrator-Paid school fees
qf5_08_4: qf5_08_4. Pension fund administrator-Paid a bill for medical treatment, housing,
qf5_08_5: qf5_08_5. Pension fund administrator-Paid a tax, fine, fee, or other payment to
qf5_08_6: qf5_08_6. Pension fund administrator-Sent money to, or received money from someo
qf5_08_7: qf5_08_7. Pension fund administrator-Received money from the government
qf5_08_8: qf5_08_8. Pension fund administrator-Received wages from an empl

In [10]:
import re

patterns = re.compile(
    r"(^id$|_id$|^id_|hhid|caseid|respondent.?id|"
    r"\bea\b|_ea|ea_|psu|cluster|strat|"
    r"enumeration|sample.?unit|segment|"
    r"household.?number|interview.?number)",
    flags=re.IGNORECASE
)

matches = []

for col in df.columns:
    label = variable_labels.get(col, "")
    if patterns.search(str(col)) or patterns.search(str(label)):
        matches.append({
            "Variable": col,
            "Label": label,
            "Unique values": df[col].nunique(dropna=True),
            "Missing values": df[col].isna().sum()
        })

pd.DataFrame(matches).to_string(index=False) if matches else \
    print("No likely sampling or identifier variables found.")

" Variable                                                                            Label  Unique values  Missing values\n   qf1_08                                   qf1_08. Ever used - Pension fund administrator              2               0\n   qf3_08              qf3_08. Account registered in own name - Pension fund administrator              2           13766\n   qf4_08                        qf4_08. Do you currently use - Pension fund administrator              2           13766\n qf5_08_1                 qf5_08_1. Pension fund administrator-Deposited or withdrew money              2           13766\n qf5_08_2 qf5_08_2. Pension fund administrator-Bought airtime top-ups or paid a mobile pho              2           13766\n qf5_08_3                            qf5_08_3. Pension fund administrator-Paid school fees              2           13766\n qf5_08_4 qf5_08_4. Pension fund administrator-Paid a bill for medical treatment, housing,              2           13766\n qf5_08_5  qf5_

In [11]:
# Inspect variables that might identify records or sampling units

for col in df.columns:
    name = str(col).lower()
    label = str(variable_labels.get(col, "")).lower()

    if any(term in name for term in [
        "id", "ea_", "psu", "cluster", "stratum",
        "hhid", "case", "serial", "enum"
    ]) or any(term in label for term in [
        "identifier", "household number", "respondent number",
        "enumeration area", "primary sampling unit"
    ]):
        print(f"{col}: {variable_labels.get(col, '')}")

key: Household identifier


In [12]:
# Check the household identifier
print("Total respondents:", len(df))
print("Missing household IDs:", df["key"].isna().sum())
print("Unique household IDs:", df["key"].nunique())
print("Households with multiple respondents:", (df["key"].value_counts() > 1).sum())

Total respondents: 13994
Missing household IDs: 0
Unique household IDs: 13994
Households with multiple respondents: 0


In [13]:
# Inspect response categories for the four indicators

variables = ["k1", "f5", "h2_1", "g3a_17"]

for col in variables:
    print(f"\n--- {col}: {variable_labels.get(col, 'No label')} ---")
    print(df[col].value_counts(dropna=False).to_string())


--- k1: K1.  you have a bank account in your name? ---
k1
No     11080
Yes     2914

--- f5: F5. Which of the following statements best describes your experience with mobile ---
f5
You are a registered mobile money user                              9434
You are not registered and have never used mobile money services    1877
You have not heard about the Mobile Money                            959
You are registered and have used it before but not any more          883
You are registered but you have never used mobile money services     452
You use mobile money services, but you are not registered            389

--- h2_1:  H2_1. Borrowed money during the past 12 months ---
h2_1
No     8193
Yes    5801

--- g3a_17: g3a. Saving with - Do not save ---
g3a_17
No     11396
Yes     2598


In [14]:

# Step 13: Calculate weighted national financial-inclusion indicators

indicators = {
    "Owns a bank account": df["k1"].astype("string").map({
        "Yes": 1, "No": 0
    }),
    "Registered mobile-money user": (
        df["f5"].astype("string")
        .eq("You are a registered mobile money user")
        .astype("int8")
    ),
    "Borrowed in the past 12 months": df["h2_1"].astype("string").map({
        "Yes": 1, "No": 0
    }),
    "Does not save": df["g3a_17"].astype("string").map({
        "Yes": 1, "No": 0
    }),
}

results = []

for name, indicator in indicators.items():
    valid = indicator.notna() & df["pop_wt"].notna()

    weighted_rate = (
        (indicator[valid] * df.loc[valid, "pop_wt"]).sum()
        / df.loc[valid, "pop_wt"].sum()
    ) * 100

    results.append({
        "Indicator": name,
        "Valid respondents": int(valid.sum()),
        "Unweighted sample (%)": round(indicator[valid].mean() * 100, 2),
        "Weighted estimate (%)": round(weighted_rate, 2),
    })

national_results = pd.DataFrame(results)
display(national_results)


,Indicator,Valid respondents,Unweighted sample (%),Weighted estimate (%)
0,Owns a bank account,13994,20.82,20.20
1,Registered mobile-money user,13994,67.41,66.61
2,Borrowed in the past 12 months,13994,41.45,39.51
3,Does not save,13994,18.57,20.25


In [15]:

# Step 14: Calculate weighted indicators by district

district_data = df[["a2", "pop_wt"]].copy()

for name, indicator in indicators.items():
    district_data[name] = indicator

district_results = []

for district, group in district_data.groupby("a2", observed=True):
    row = {"District": district}

    for name in indicators:
        valid = group[name].notna() & group["pop_wt"].notna()
        valid_data = group.loc[valid]

        row[f"{name} (%)"] = (
            (
                valid_data[name] * valid_data["pop_wt"]
            ).sum() / valid_data["pop_wt"].sum() * 100
            if len(valid_data) > 0 else float("nan")
        )

    row["Valid respondents"] = len(group)
    district_results.append(row)

district_results = (
    pd.DataFrame(district_results)
    .sort_values("District")
    .round(2)
    .reset_index(drop=True)
)

display(district_results)


,District,Owns a bank account (%),Registered mobile-money user (%),Borrowed in the past 12 months (%),Does not save (%),Valid respondents
0,Bugesera,21.21,72.08,50.31,11.44,560
1,Burera,9.58,73.25,34.94,25.15,416
2,Gakenke,17.35,56.37,30.95,27.93,447
3,Gasabo,47.17,88.07,48.17,10.44,848
4,Gatsibo,20.98,62.78,48.37,11.79,400
5,Gicumbi,6.51,52.37,39.06,19.60,512
6,Gisagara,9.53,54.57,29.01,22.38,400
7,Huye,13.92,54.38,24.93,30.63,480
8,Kamonyi,15.20,66.82,39.05,16.02,623
9,Karongi,7.19,47.87,35.35,21.33,399


In [16]:
# Investigate bank account ownership in Kirehe

kirehe = df[df["a2"].astype("string").eq("Kirehe")].copy()

print("Kirehe respondents:", len(kirehe))
print("\nUnweighted responses:")
print(kirehe["k1"].value_counts())

print("\nUnweighted bank ownership (%):",
      round(kirehe["k1"].eq("Yes").mean() * 100, 2))

print("\nWeighted bank ownership (%):",
      round(
          (
              kirehe["k1"].eq("Yes").astype(float) * kirehe["pop_wt"]
          ).sum() / kirehe["pop_wt"].sum() * 100,
          2
      ))

Kirehe respondents: 464

Unweighted responses:
k1
No     428
Yes     36
Name: count, dtype: int64

Unweighted bank ownership (%): 7.76

Weighted bank ownership (%): 5.41


In [17]:

# Step 15: Summarize the lowest and highest district estimates

indicator_columns = [
    "Owns a bank account (%)",
    "Registered mobile-money user (%)",
    "Borrowed in the past 12 months (%)",
    "Does not save (%)",
]

gap_summary = []

for col in indicator_columns:
    lowest = district_results.loc[district_results[col].idxmin()]
    highest = district_results.loc[district_results[col].idxmax()]

    gap_summary.append({
        "Indicator": col.replace(" (%)", ""),
        "Lowest district": lowest["District"],
        "Lowest estimate (%)": lowest[col],
        "Highest district": highest["District"],
        "Highest estimate (%)": highest[col],
        "Difference (percentage points)": round(
            highest[col] - lowest[col], 2
        ),
    })

gap_summary = pd.DataFrame(gap_summary)
display(gap_summary)


,Indicator,Lowest district,Lowest estimate (%),Highest district,Highest estimate (%),Difference (percentage points)
0,Owns a bank account,Kirehe,5.41,Kicukiro,59.51,54.10
1,Registered mobile-money user,Nyaruguru,45.96,Kicukiro,89.03,43.07
2,Borrowed in the past 12 months,Ngororero,14.94,Nyanza,76.43,61.49
3,Does not save,Musanze,8.40,Nyaruguru,46.56,38.16


In [18]:

# Step 16: Compare financial-inclusion indicators by urban/rural residence

urban_rural_data = df[["a6", "pop_wt"]].copy()

for name, indicator in indicators.items():
    urban_rural_data[name] = indicator

urban_rural_results = []

for area, group in urban_rural_data.groupby("a6", observed=True):
    row = {"Area": area, "Valid respondents": len(group)}

    for name in indicators:
        valid = group[name].notna() & group["pop_wt"].notna()
        valid_data = group.loc[valid]

        row[f"{name} (%)"] = round(
            (
                (valid_data[name] * valid_data["pop_wt"]).sum()
                / valid_data["pop_wt"].sum()
            ) * 100,
            2
        )

    urban_rural_results.append(row)

urban_rural_results = pd.DataFrame(urban_rural_results)
display(urban_rural_results)


,Area,Valid respondents,Owns a bank account (%),Registered mobile-money user (%),Borrowed in the past 12 months (%),Does not save (%)
0,Urban,5789,37.06,81.26,37.59,17.69
1,Rural,8205,12.64,60.04,40.37,21.39


In [19]:
from pathlib import Path

output_dir = project_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

district_results.to_csv(
    output_dir / "district_financial_inclusion.csv",
    index=False
)

urban_rural_results.to_csv(
    output_dir / "urban_rural_financial_inclusion.csv",
    index=False
)

gap_summary.to_csv(
    output_dir / "district_gap_summary.csv",
    index=False
)

print("Saved dashboard-ready indicator tables to:", output_dir)

Saved dashboard-ready indicator tables to: /home/ibrahim/Desktop/Backup/PPioneers/nisr-big-data-hackathon-project/data/processed


In [20]:

# Step 3.3: Compare financial inclusion across age groups

age_data = df[["b1", "pop_wt"]].copy()

# Define age groups
age_data["Age group"] = pd.cut(
    age_data["b1"].astype(float),
    bins=[15, 24, 34, 44, 54, 64, float("inf")],
    labels=["16–24", "25–34", "35–44", "45–54", "55–64", "65+"]
)

# Add the four existing indicators
for name, indicator in indicators.items():
    age_data[name] = indicator

age_results = []

for age_group, group in age_data.groupby("Age group", observed=True):
    row = {
        "Age group": age_group,
        "Valid respondents": len(group)
    }

    for name in indicators:
        valid = group[name].notna() & group["pop_wt"].notna()
        valid_data = group.loc[valid]

        row[f"{name} (%)"] = round(
            (
                (valid_data[name] * valid_data["pop_wt"]).sum()
                / valid_data["pop_wt"].sum()
            ) * 100,
            2
        )

    age_results.append(row)

age_results = pd.DataFrame(age_results)

display(age_results)


,Age group,Valid respondents,Owns a bank account (%),Registered mobile-money user (%),Borrowed in the past 12 months (%),Does not save (%)
0,16–24,1777,14.37,58.52,27.51,32.75
1,25–34,2900,24.32,77.81,46.93,12.45
2,35–44,3788,24.77,73.48,46.16,12.23
3,45–54,2180,22.82,70.54,45.68,13.29
4,55–64,1548,18.97,59.93,43.39,20.40
5,65+,1801,14.61,45.57,30.34,28.94


In [21]:

# Step 3.3: Compare financial inclusion by sex

sex_data = df[["b2", "pop_wt"]].copy()

for name, indicator in indicators.items():
    sex_data[name] = indicator

sex_results = []

for sex, group in sex_data.groupby("b2", observed=True):
    row = {
        "Sex": sex,
        "Valid respondents": len(group)
    }

    for name in indicators:
        valid = group[name].notna() & group["pop_wt"].notna()
        valid_data = group.loc[valid]

        row[f"{name} (%)"] = round(
            (
                (valid_data[name] * valid_data["pop_wt"]).sum()
                / valid_data["pop_wt"].sum()
            ) * 100,
            2
        )

    sex_results.append(row)

sex_results = pd.DataFrame(sex_results)

display(sex_results)


,Sex,Valid respondents,Owns a bank account (%),Registered mobile-money user (%),Borrowed in the past 12 months (%),Does not save (%)
0,Male,6186,25.32,72.16,39.20,18.72
1,Female,7808,15.48,61.50,39.79,21.66


In [22]:

# Step 3.3: Compare financial inclusion by education

education_data = df[["c4a", "pop_wt"]].copy()

for name, indicator in indicators.items():
    education_data[name] = indicator

education_results = []

for education, group in education_data.groupby("c4a", observed=True):
    row = {
        "Education": education,
        "Valid respondents": len(group)
    }

    for name in indicators:
        valid = group[name].notna() & group["pop_wt"].notna()
        valid_data = group.loc[valid]

        row[f"{name} (%)"] = round(
            (
                (valid_data[name] * valid_data["pop_wt"]).sum()
                / valid_data["pop_wt"].sum()
            ) * 100,
            2
        )

    education_results.append(row)

education_results = pd.DataFrame(education_results)

display(education_results)


,Education,Valid respondents,Owns a bank account (%),Registered mobile-money user (%),Borrowed in the past 12 months (%),Does not save (%)
0,No formal education,2510,6.04,41.37,32.48,26.89
1,Primary 1-3,1875,10.20,55.27,39.47,20.86
2,Primary 4-6,5675,13.88,67.94,43.05,18.50
3,Secondary 1-3,1395,18.43,66.95,35.41,25.55
4,Secondary 4-6,1596,40.99,84.52,38.93,19.46
5,University or other higher education,799,78.68,98.48,43.29,5.19
6,Vocational training,144,34.95,92.59,37.33,10.55


In [24]:

# Step 3.3: Compare financial inclusion by marital status

marital_data = df[["c4b", "pop_wt"]].copy()

for name, indicator in indicators.items():
    marital_data[name] = indicator

marital_results = []

for status, group in marital_data.groupby("c4b", observed=True):
    row = {
        "Marital status": status,
        "Valid respondents": len(group)
    }

    for name in indicators:
        valid = group[name].notna() & group["pop_wt"].notna()
        valid_data = group.loc[valid]

        row[f"{name} (%)"] = round(
            (
                (valid_data[name] * valid_data["pop_wt"]).sum()
                / valid_data["pop_wt"].sum()
            ) * 100,
            2
        )

    marital_results.append(row)

marital_results = pd.DataFrame(marital_results)

display(marital_results)


,Marital status,Valid respondents,Owns a bank account (%),Registered mobile-money user (%),Borrowed in the past 12 months (%),Does not save (%)
0,Never married,2478,19.28,63.63,29.81,28.17
1,Married,7638,20.85,68.06,44.33,16.96
2,Living together,1909,20.03,68.68,41.70,17.65
3,Divorced/Separated,594,22.51,66.66,40.98,15.30
4,Widowed,1375,18.44,64.01,37.07,21.49


In [25]:

# Find variables whose labels mention employment or work

employment_terms = ["employment", "employed", "occupation", "work status", "working"]

employment_candidates = {
    var: label
    for var, label in variable_labels.items()
    if any(term in str(label).lower() for term in employment_terms)
}

for var, label in employment_candidates.items():
    print(f"{var}: {label}")

print(f"\nTotal candidates: {len(employment_candidates)}")


qf6_01_4: qf6_01_4.Commercial bank-ATM or Card Swipe machine not working
qf6_02_4: qf6_02_4.Digital bank-ATM or Card Swipe machine not working
qf6_05_4: qf6_05_4.Mobile money operator-ATM or Card Swipe machine not working
qf6_06_4: qf6_06_4.Umurenge SACCO-ATM or Card Swipe machine not working
qf6_07_4: qf6_07_4.Insurance provider-ATM or Card Swipe machine not working
qf6_08_4: qf6_08_4.Pension fund administrator-ATM or Card Swipe machine not working
qf6_09_4: qf6_09_4.Long-Term Saving Scheme (EJOHEZA)-ATM or Card Swipe machine not working
qf6_11_4: qf6_11_4.Village/community association-ATM or Card Swipe machine not working
n1a_5: n1a_5. Self-employed (have own registered/licensed business)
n1a_22: n1a_22. Self-employed (have own business -not registered/licensed)
n4: N4. Which of these, if any, best describes your personal working status?
n5: N5. How long you have been unemployed?
n13_2: n13_2. Stopped working for some time

Total candidates: 13


In [26]:

print("Variable label:")
print(variable_labels.get("n4", "Label not found"))

print("\nWorking-status response counts:")
display(
    df["n4"]
    .value_counts(dropna=False)
    .rename_axis("Working status")
    .reset_index(name="Respondents")
)


Variable label:
N4. Which of these, if any, best describes your personal working status?

Working-status response counts:


,Working status,Respondents
0,Work full-time or 30 hours a week or more for ...,3140
1,Housewife or house husband,2840
2,Work full-time or 30 hours a week or more for ...,2832
3,Work part-time or less than 30 hours a week fo...,1400
4,Work part-time or less than 30 hours a week fo...,1168
5,Work part-time or less than 8 hours a week for...,963
6,Student or learner (in full-time education),386
7,Unemployed and not looking for a job,323
8,Long term illness / disabled,320
9,Pensioner or retired,177


In [27]:

# Step 3.3: Compare financial inclusion by working status

work_data = df[["n4", "pop_wt"]].copy()

for name, indicator in indicators.items():
    work_data[name] = indicator

work_results = []

for status, group in work_data.groupby("n4", observed=True):
    row = {
        "Working status": status,
        "Valid respondents": len(group)
    }

    for name in indicators:
        valid = group[name].notna() & group["pop_wt"].notna()
        valid_data = group.loc[valid]

        row[f"{name} (%)"] = round(
            (
                (valid_data[name] * valid_data["pop_wt"]).sum()
                / valid_data["pop_wt"].sum()
            ) * 100,
            2
        )

    work_results.append(row)

work_results = pd.DataFrame(work_results)

display(work_results)


,Working status,Valid respondents,Owns a bank account (%),Registered mobile-money user (%),Borrowed in the past 12 months (%),Does not save (%)
0,Work full-time or 30 hours a week or more for ...,2832,28.78,79.57,48.90,7.84
1,Work full-time or 30 hours a week or more for ...,3140,28.16,72.61,46.51,13.71
2,Work part-time or less than 30 hours a week fo...,1168,19.70,73.99,47.10,10.16
3,Work part-time or less than 30 hours a week fo...,1400,15.78,61.52,50.23,15.18
4,Work part-time or less than 8 hours a week for...,963,16.09,67.68,48.40,14.46
5,Student or learner (in full-time education),386,3.69,35.31,9.12,64.21
6,Housewife or house husband,2840,9.59,58.81,24.51,29.01
7,Pensioner or retired,177,43.48,79.05,22.55,19.89
8,Long term illness / disabled,320,7.13,35.83,23.92,44.56
9,Unemployed and looking for first job,162,19.95,68.97,21.57,39.51


In [28]:

# Step 3.4: Construct transparent district-level gap indicators

district_gaps = district_results[
    ["District", "Valid respondents"]
].copy()

gap_indicators = {
    "Bank-account ownership gap (pp)":
        "Owns a bank account (%)",
    "Mobile-money registration gap (pp)":
        "Registered mobile-money user (%)",
    "Borrowing activity gap (pp)":
        "Borrowed in the past 12 months (%)",
}

for gap_name, inclusion_column in gap_indicators.items():
    district_gaps[gap_name] = (
        100 - district_results[inclusion_column]
    ).round(2)

display(district_gaps)


,District,Valid respondents,Bank-account ownership gap (pp),Mobile-money registration gap (pp),Borrowing activity gap (pp)
0,Bugesera,560,78.79,27.92,49.69
1,Burera,416,90.42,26.75,65.06
2,Gakenke,447,82.65,43.63,69.05
3,Gasabo,848,52.83,11.93,51.83
4,Gatsibo,400,79.02,37.22,51.63
5,Gicumbi,512,93.49,47.63,60.94
6,Gisagara,400,90.47,45.43,70.99
7,Huye,480,86.08,45.62,75.07
8,Kamonyi,623,84.80,33.18,60.95
9,Karongi,399,92.81,52.13,64.65


In [29]:

print("District results columns:")
print(district_results.columns.tolist())

print("\nFirst 5 district inclusion estimates:")
display(district_results.head())

print("\nFirst 5 calculated gaps:")
display(district_gaps.head())


District results columns:
['District', 'Owns a bank account (%)', 'Registered mobile-money user (%)', 'Borrowed in the past 12 months (%)', 'Does not save (%)', 'Valid respondents']

First 5 district inclusion estimates:


,District,Owns a bank account (%),Registered mobile-money user (%),Borrowed in the past 12 months (%),Does not save (%),Valid respondents
0,Bugesera,21.21,72.08,50.31,11.44,560
1,Burera,9.58,73.25,34.94,25.15,416
2,Gakenke,17.35,56.37,30.95,27.93,447
3,Gasabo,47.17,88.07,48.17,10.44,848
4,Gatsibo,20.98,62.78,48.37,11.79,400



First 5 calculated gaps:


,District,Valid respondents,Bank-account ownership gap (pp),Mobile-money registration gap (pp),Borrowing activity gap (pp)
0,Bugesera,560,78.79,27.92,49.69
1,Burera,416,90.42,26.75,65.06
2,Gakenke,447,82.65,43.63,69.05
3,Gasabo,848,52.83,11.93,51.83
4,Gatsibo,400,79.02,37.22,51.63


In [30]:

# Save the district gap indicators for the dashboard

gap_path = output_dir / "district_gap_indicators.csv"

district_gaps.to_csv(gap_path, index=False)

print("Saved:", gap_path)
print("Districts included:", len(district_gaps))
print("Columns:", district_gaps.columns.tolist())


Saved: /home/ibrahim/Desktop/Backup/PPioneers/nisr-big-data-hackathon-project/data/processed/district_gap_indicators.csv
Districts included: 30
Columns: ['District', 'Valid respondents', 'Bank-account ownership gap (pp)', 'Mobile-money registration gap (pp)', 'Borrowing activity gap (pp)']


In [31]:
# Save transparent definitions for the dashboard

indicator_definitions = pd.DataFrame([
    {
        "Indicator": "Owns a bank account",
        "Source variable": "k1",
        "Definition": "Weighted percentage of respondents who answered Yes.",
        "Interpretation": "Higher values indicate greater bank-account ownership."
    },
    {
        "Indicator": "Registered mobile-money user",
        "Source variable": "f5",
        "Definition": "Weighted percentage classified as a registered mobile-money user.",
        "Interpretation": "Measures registration, not necessarily current activity."
    },
    {
        "Indicator": "Borrowed in the past 12 months",
        "Source variable": "h2_1",
        "Definition": "Weighted percentage who reported borrowing in the past 12 months.",
        "Interpretation": "Measures recent borrowing activity, not formal credit access."
    },
    {
        "Indicator": "Does not save",
        "Source variable": "g3a_17",
        "Definition": "Weighted percentage who answered Yes to the do-not-save category.",
        "Interpretation": "A financial vulnerability signal, not a savings-access gap."
    },
    {
        "Indicator": "Bank-account ownership gap",
        "Source variable": "k1",
        "Definition": "100 minus the weighted bank-account ownership estimate.",
        "Interpretation": "Percentage not reporting bank-account ownership."
    },
    {
        "Indicator": "Mobile-money registration gap",
        "Source variable": "f5",
        "Definition": "100 minus the weighted registered mobile-money estimate.",
        "Interpretation": "Percentage not classified as a registered mobile-money user."
    },
    {
        "Indicator": "Borrowing activity gap",
        "Source variable": "h2_1",
        "Definition": "100 minus the weighted recent-borrowing estimate.",
        "Interpretation": "Percentage not reporting borrowing in the past 12 months."
    }
])

definitions_path = output_dir / "indicator_definitions.csv"
indicator_definitions.to_csv(definitions_path, index=False)

print("Saved:", definitions_path)
display(indicator_definitions)

Saved: /home/ibrahim/Desktop/Backup/PPioneers/nisr-big-data-hackathon-project/data/processed/indicator_definitions.csv


,Indicator,Source variable,Definition,Interpretation
0,Owns a bank account,k1,Weighted percentage of respondents who answere...,Higher values indicate greater bank-account ow...
1,Registered mobile-money user,f5,Weighted percentage classified as a registered...,"Measures registration, not necessarily current..."
2,Borrowed in the past 12 months,h2_1,Weighted percentage who reported borrowing in ...,"Measures recent borrowing activity, not formal..."
3,Does not save,g3a_17,Weighted percentage who answered Yes to the do...,"A financial vulnerability signal, not a saving..."
4,Bank-account ownership gap,k1,100 minus the weighted bank-account ownership ...,Percentage not reporting bank-account ownership.
5,Mobile-money registration gap,f5,100 minus the weighted registered mobile-money...,Percentage not classified as a registered mobi...
6,Borrowing activity gap,h2_1,100 minus the weighted recent-borrowing estimate.,Percentage not reporting borrowing in the past...
